# Qwen3-8B 大模型部署与推理教程

## 学习目标
1. 了解如何下载和加载大语言模型
2. 掌握使用 Transformers 库进行模型推理
3. 理解 Qwen3 的思考模式（Thinking Mode）特性

## 环境要求
- GPU: 至少 16GB 显存（如 Tesla T4）
- Python 3.8+
- CUDA 支持

## 参考教程
- 原始教程：[datawhalechina/self-llm](https://github.com/datawhalechina/self-llm/blob/master/models/Qwen3/02-Qwen3-8B-vLLM%20部署调用.md)

In [ ]:
# ============================================================
# 第一步：安装必要的 Python 库
# ============================================================
# 
# 【语法说明】
# - 在 Jupyter Notebook 中，! 开头的命令会在系统终端（Shell）中执行
# - pip 是 Python 的包管理工具，用于安装第三方库
# - install 是 pip 的子命令，表示安装
#
# 【安装的库说明】
# 1. modelscope: 阿里巴巴的模型库平台，类似于 Hugging Face
#    - 用于从国内服务器下载模型，速度更快
#    - 官网：https://www.modelscope.cn
#
# 2. vllm: 高性能 LLM 推理引擎（本教程最终使用 Transformers 替代）
#    - 支持高吞吐量推理
#    - 兼容 OpenAI API 格式
# ============================================================

!pip install modelscope
!pip install vllm

In [ ]:
# ============================================================
# 第二步：从 ModelScope 下载 Qwen3-8B 模型
# ============================================================
#
# 【导入语句说明】
# from 模块名 import 函数名
# - 从 modelscope 库中只导入 snapshot_download 这一个函数
# - 这样可以直接使用 snapshot_download()，而不需要写 modelscope.snapshot_download()
# ============================================================

from modelscope import snapshot_download

# ============================================================
# 【函数调用说明】
# snapshot_download() 函数用于下载模型文件
#
# 参数解释：
# 1. 'Qwen/Qwen3-8B' (第一个参数，位置参数)
#    - 模型的唯一标识符，格式为 "组织名/模型名"
#    - Qwen 是阿里通义千问团队
#    - Qwen3-8B 表示 Qwen 第三代，80亿参数版本
#
# 2. cache_dir='/root/autodl-tmp' (关键字参数)
#    - 指定模型下载后保存的目录路径
#    - /root/autodl-tmp 是 AutoDL 平台的数据盘路径
#    - 如果使用其他平台，需要修改为对应的路径
#
# 3. revision='master' (关键字参数)
#    - 指定要下载的模型版本/分支
#    - 'master' 表示主分支，即最新稳定版
#
# 返回值：
# - model_dir: 返回模型文件实际保存的完整路径
# ============================================================

model_dir = snapshot_download('Qwen/Qwen3-8B', cache_dir='/root/autodl-tmp', revision='master')

# 下载完成后，模型文件会保存在：/root/autodl-tmp/Qwen/Qwen3-8B/
# 包含以下文件：
# - config.json: 模型配置文件
# - model-00001-of-00005.safetensors 等: 模型权重文件（分片存储）
# - tokenizer.json: 分词器配置
# - vocab.json: 词汇表

---

## 方案说明

由于 vLLM 最新版本在 Colab Notebook 环境中存在多进程兼容性问题，我们改用 **Transformers + PyTorch** 进行推理。

### Transformers vs vLLM 对比

| 特性 | Transformers | vLLM |
|------|-------------|------|
| 易用性 | 简单，适合学习 | 需要更多配置 |
| 性能 | 一般 | 高吞吐量 |
| 兼容性 | 好，支持各种环境 | Notebook 中有问题 |
| 适用场景 | 学习、小规模推理 | 生产部署、高并发 |

### 核心概念

1. **Tokenizer（分词器）**：将文本转换为模型能理解的数字序列
2. **Model（模型）**：神经网络，处理数字序列并生成输出
3. **Generation（生成）**：模型根据输入逐步生成新的 token

In [9]:
# ============================================================
# 第三步：使用 Transformers 加载模型
# ============================================================
#
# 【导入库说明】
# import 库名: 导入整个库，使用时需要 库名.xxx
# from 库名 import xxx: 只导入特定的类/函数
# ============================================================

import torch  
# torch 是 PyTorch 库的核心模块
# PyTorch 是深度学习框架，提供张量计算和 GPU 加速
# 这里主要用于指定数据类型（float16）

from transformers import AutoModelForCausalLM, AutoTokenizer
# transformers 是 Hugging Face 开发的 NLP 库
# 
# AutoTokenizer: 自动加载分词器的类
#   - "Auto" 前缀表示会根据模型配置自动选择正确的分词器类型
#   - 分词器负责：文本 → 数字序列（编码）、数字序列 → 文本（解码）
#
# AutoModelForCausalLM: 自动加载因果语言模型的类
#   - "CausalLM" = Causal Language Model（因果语言模型）
#   - 因果语言模型：只能看到当前位置之前的内容，用于文本生成
#   - GPT、Qwen、LLaMA 都属于这类模型

# ============================================================
# 【变量定义】
# Python 中变量不需要声明类型，直接赋值即可
# ============================================================

model_path = '/root/autodl-tmp/Qwen/Qwen3-8B'
# model_path: 字符串变量，存储模型文件所在的目录路径
# 这个路径就是前面 snapshot_download 下载模型的位置

# ============================================================
# 【加载分词器】
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(model_path)
# from_pretrained() 是一个类方法（class method）
# 作用：从指定路径加载预训练的分词器
# 
# 参数：model_path - 模型目录路径，会自动读取其中的 tokenizer.json 等文件
# 返回值：tokenizer 对象，可以用于文本的编码和解码
#
# 分词器的主要功能：
# 1. tokenizer.encode("文本") → [数字列表]  将文本转为 token ID
# 2. tokenizer.decode([数字列表]) → "文本"  将 token ID 转回文本
# 3. tokenizer.apply_chat_template() → 格式化对话

print("分词器加载完成！")
# print() 是 Python 内置函数，用于在控制台输出信息

# ============================================================
# 【加载模型】
# ============================================================

model = AutoModelForCausalLM.from_pretrained(
    model_path,                    # 参数1：模型路径
    torch_dtype=torch.float16,     # 参数2：数据类型
    device_map="auto"              # 参数3：设备映射
)
# 
# 【参数详解】
#
# 1. model_path (位置参数)
#    - 模型文件所在目录
#    - 会读取 config.json 和 model-*.safetensors 文件
#
# 2. torch_dtype=torch.float16 (关键字参数)
#    - 指定模型权重的数据类型
#    - float16（半精度）：每个数字用 16 位存储
#    - float32（单精度）：每个数字用 32 位存储
#    - 使用 float16 可以节省一半显存，但精度略有损失
#    - 对于推理来说，float16 足够用
#
# 3. device_map="auto" (关键字参数)
#    - 自动将模型分配到可用的设备（GPU/CPU）
#    - "auto" 会优先使用 GPU，显存不足时会部分放到 CPU
#    - 也可以手动指定：device_map="cuda:0" (第一块 GPU)
#
# 【返回值】
# model: 一个 PyTorch 神经网络模型对象
# - 包含约 80 亿个参数（8B = 8 Billion）
# - 使用 float16 时约占用 16GB 显存

print("模型加载完成！")

# ============================================================
# 【警告说明】
# 运行时可能看到以下警告，可以忽略：
# 
# 1. "torch_dtype is deprecated! Use dtype instead!"
#    - 这是一个弃用警告，不影响运行
#    - 未来版本可能需要改用 dtype 参数
#
# 2. "Some parameters are on the meta device because they were offloaded to the cpu"
#    - 表示部分模型参数被放到了 CPU（因为 GPU 显存不够）
#    - 这会稍微影响推理速度，但功能正常
# ============================================================

分词器加载完成！


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

模型加载完成！


In [10]:
# ============================================================
# 第四步：准备输入 - 使用 Chat Template 格式化对话
# ============================================================
#
# 【背景知识】
# 大语言模型需要特定的输入格式才能正确理解对话
# 不同模型有不同的格式，例如：
# - ChatGPT 格式: [{"role": "user", "content": "..."}]
# - Qwen 格式: <|im_start|>user\n...<|im_end|>\n<|im_start|>assistant\n
#
# apply_chat_template() 会自动处理这些格式转换
# ============================================================

# 【定义用户输入】
prompt = "给我一个关于大模型的简短介绍。"
# prompt: 字符串变量，存储用户的问题/指令

# 【构建对话消息列表】
messages = [{"role": "user", "content": prompt}]
# messages: 列表（list），包含一个字典（dict）
#
# Python 数据结构说明：
# - 列表 []: 有序的元素集合，用方括号表示
# - 字典 {}: 键值对集合，用花括号表示，格式为 {键: 值}
#
# 这里的字典包含两个键值对：
# - "role": "user"     表示这是用户说的话
# - "content": prompt  表示具体内容
#
# 如果是多轮对话，messages 会包含多个字典：
# messages = [
#     {"role": "user", "content": "你好"},
#     {"role": "assistant", "content": "你好！有什么可以帮助你的？"},
#     {"role": "user", "content": "介绍一下大模型"}
# ]

# ============================================================
# 【使用 Chat Template 格式化输入】
# ============================================================

text = tokenizer.apply_chat_template(
    messages,                      # 参数1：对话消息列表
    tokenize=False,                # 参数2：是否进行分词
    add_generation_prompt=True,    # 参数3：是否添加生成提示
    enable_thinking=True           # 参数4：是否开启思考模式（Qwen3 特有）
)
#
# 【参数详解】
#
# 1. messages (位置参数)
#    - 对话消息列表，格式如上所述
#
# 2. tokenize=False (关键字参数)
#    - False: 返回格式化后的字符串（人类可读）
#    - True: 返回 token ID 列表（数字序列）
#    - 这里用 False 是为了方便查看格式化结果
#
# 3. add_generation_prompt=True (关键字参数)
#    - True: 在末尾添加 assistant 的开始标记
#    - 这告诉模型"现在该你回答了"
#    - 如果不加，模型可能不知道该生成回复
#
# 4. enable_thinking=True (关键字参数) ⭐ Qwen3 特有功能
#    - True: 开启思考模式，模型会先展示推理过程，再给出答案
#    - False: 直接输出答案，不显示思考过程
#    - 思考过程会用 <think>...</think> 标签包裹
#    - 类似于 DeepSeek-R1 和 QwQ 的推理能力
#
# 【返回值】
# text: 格式化后的字符串，可以直接输入给模型

# 【打印查看格式化结果】
print("格式化后的输入：")
print(text)
print("=" * 50)
# "=" * 50 是字符串重复操作，生成 50 个等号组成的分隔线

# ============================================================
# 【格式化结果解释】
# 输出类似于：
# <|im_start|>user
# 给我一个关于大模型的简短介绍。<|im_end|>
# <|im_start|>assistant
#
# 各部分含义：
# - <|im_start|>: 消息开始标记（im = instant message）
# - user / assistant: 角色标识
# - <|im_end|>: 消息结束标记
# - 最后的 assistant 后面没有 <|im_end|>，表示等待模型生成内容
# ============================================================

格式化后的输入：
<|im_start|>user
给我一个关于大模型的简短介绍。<|im_end|>
<|im_start|>assistant



In [11]:
# ============================================================
# 第五步：执行模型推理（文本生成）
# ============================================================
#
# 【推理流程概述】
# 1. 将文本编码为 token ID（数字序列）
# 2. 将数据移动到 GPU
# 3. 调用模型的 generate() 方法生成新 token
# 4. 将生成的 token ID 解码回文本
# ============================================================

# ============================================================
# 【第一步：文本编码】
# ============================================================

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
#
# 这行代码做了三件事，让我们逐一解析：
#
# 【1】tokenizer([text], return_tensors="pt")
#     - tokenizer 对象可以直接当函数调用
#     - [text]: 将 text 放入列表中，因为 tokenizer 支持批量处理
#     - return_tensors="pt": 返回 PyTorch 张量格式
#       - "pt" = PyTorch
#       - "tf" = TensorFlow
#       - "np" = NumPy
#     - 返回一个字典，包含：
#       - input_ids: token ID 序列，如 [151644, 8948, ...]
#       - attention_mask: 注意力掩码，标记哪些位置是有效的
#
# 【2】.to(model.device)
#     - .to() 是 PyTorch 张量的方法，用于移动数据到指定设备
#     - model.device 获取模型所在的设备（如 cuda:0）
#     - 模型和数据必须在同一设备上才能计算
#
# 【3】结果赋值给 model_inputs
#     - model_inputs 是一个类字典对象
#     - 可以用 model_inputs.input_ids 访问 token ID
#     - 可以用 model_inputs['input_ids'] 也可以

# ============================================================
# 【第二步：调用模型生成】
# ============================================================

generated_ids = model.generate(
    **model_inputs,         # 输入数据（使用 ** 解包字典）
    max_new_tokens=2048,    # 最大生成 token 数
    temperature=0.6,        # 温度参数
    top_p=0.95,             # 核采样参数
    top_k=20,               # Top-K 采样参数
    do_sample=True          # 是否使用采样
)
#
# 【参数详解】
#
# 1. **model_inputs (解包操作)
#    - ** 是 Python 的字典解包操作符
#    - 相当于把字典中的键值对展开为关键字参数
#    - 等价于：model.generate(input_ids=..., attention_mask=...)
#
# 2. max_new_tokens=2048
#    - 限制生成的最大 token 数量
#    - 防止模型无限生成下去
#    - 2048 个 token 大约是 1000-2000 个汉字
#
# 3. temperature=0.6 ⭐ 重要参数
#    - 控制生成的随机性/创造性
#    - 范围：0 到 2（通常用 0.1-1.0）
#    - 低温度（如 0.1）：输出更确定、更保守
#    - 高温度（如 1.0）：输出更随机、更有创意
#    - 0.6 是思考模式的官方推荐值
#
# 4. top_p=0.95 (核采样 / Nucleus Sampling)
#    - 只考虑概率累计达到 top_p 的 token
#    - 0.95 表示只从概率最高的 95% token 中采样
#    - 可以过滤掉低概率的"噪音" token
#
# 5. top_k=20 (Top-K 采样)
#    - 只考虑概率最高的 K 个 token
#    - 20 表示每一步只从概率前 20 的 token 中选择
#    - 与 top_p 配合使用，进一步控制采样范围
#
# 6. do_sample=True
#    - True: 使用采样（随机选择），输出有变化
#    - False: 使用贪婪解码（选概率最高的），输出固定
#    - 使用 temperature/top_p/top_k 时必须设为 True
#
# 【返回值】
# generated_ids: 形状为 [batch_size, sequence_length] 的张量
# 包含输入 token + 生成的新 token

# ============================================================
# 【第三步：提取生成的部分】
# ============================================================

output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
#
# 让我们拆解这行代码：
#
# 1. generated_ids[0]
#    - generated_ids 形状是 [1, 总长度]（因为只有一个输入）
#    - [0] 取第一个（也是唯一一个）样本
#
# 2. [len(model_inputs.input_ids[0]):]
#    - 这是 Python 的切片操作
#    - len(model_inputs.input_ids[0]) 是输入的长度
#    - [n:] 表示从位置 n 开始取到末尾
#    - 效果：去掉输入部分，只保留新生成的 token
#
# 3. .tolist()
#    - 将 PyTorch 张量转换为 Python 列表
#    - 方便后续处理

# ============================================================
# 【第四步：解码为文本】
# ============================================================

response = tokenizer.decode(output_ids, skip_special_tokens=True)
#
# tokenizer.decode() 将 token ID 转换回文本
#
# 参数：
# - output_ids: token ID 列表
# - skip_special_tokens=True: 跳过特殊 token（如 <|im_end|>）
#
# 返回值：解码后的文本字符串

# ============================================================
# 【输出结果】
# ============================================================

print("=" * 50)
print("模型响应（包含思考过程）：")
print("=" * 50)
print(response)

# ============================================================
# 【输出格式说明】
# 由于开启了思考模式，输出会包含两部分：
#
# 1. <think>...</think> 部分
#    - 模型的内部推理过程
#    - 展示模型是如何思考这个问题的
#    - 类似于"草稿纸"
#
# 2. </think> 之后的部分
#    - 最终给用户的回答
#    - 经过思考后的精炼答案
#
# 这种"思考-回答"模式让模型能处理更复杂的问题
# ============================================================

模型响应（包含思考过程）：
<think>
好的，用户让我给一个关于大模型的简短介绍。首先，我需要确定用户的需求是什么。他们可能是在做研究，或者写报告，或者只是想了解大模型的基本概念。简短介绍的话，应该涵盖定义、特点、应用场景和影响这几个方面。

接下来，我得考虑用户可能的背景。如果是学生，可能需要更基础的解释；如果是专业人士，可能需要更深入的技术细节。但用户要求的是简短，所以得保持简洁，避免太专业的术语，同时也要准确。

然后，我需要确保涵盖大模型的关键点。比如，大模型是基于深度学习的，参数量巨大，通过大量数据训练，具备强大的语言理解和生成能力。应用场景包括自然语言处理、机器翻译、文本生成等。还要提到它们在各行业的应用，比如客服、内容创作、数据分析等。

另外，用户可能想知道大模型的影响，比如推动AI发展，但也可能有挑战，比如计算资源需求高、数据隐私问题。不过用户要的是简短介绍，可能不需要深入讨论挑战，但可以稍微提一下。

还要注意结构清晰，分点或分段，但用户要求的是简短，所以可能需要一段式。不过用户给的例子是分段的，所以可能更合适。需要检查是否有冗余信息，确保每个部分都简洁明了。

最后，确保语言流畅，用词准确，避免错误。比如，大模型的参数量通常在十亿到万亿级别，这点要准确。同时，应用场景要具体，比如智能客服、内容创作、数据分析等，这些都是常见的例子。

总结一下，结构应该是：定义，核心技术，特点，应用场景，影响和挑战。保持每个部分简短，用一两句话说明。这样用户就能快速理解大模型的基本概念和重要性。
</think>

大模型（Large Language Models, LLMs）是基于深度学习技术训练而成的复杂人工智能系统，通常拥有数十亿至数万亿参数，通过海量文本数据学习语言规律，具备强大的自然语言理解、生成和推理能力。其核心特点包括：**多任务处理**（如翻译、问答、编程）、**上下文感知**（理解对话历史）和**跨领域适应性**（可迁移至不同场景）。应用场景涵盖智能客服、内容创作、数据分析、科学研究等，正在深刻改变人工智能技术的边界与落地方式。同时，其高算力需求和数据依赖性也带来技术挑战。


---

## 关闭思考模式测试

### 思考模式 vs 非思考模式对比

| 特性 | 思考模式 (enable_thinking=True) | 非思考模式 (enable_thinking=False) |
|------|-------------------------------|----------------------------------|
| 输出内容 | `<think>推理过程</think>` + 答案 | 直接输出答案 |
| 推荐温度 | temperature=0.6 | temperature=0.7 |
| 推荐 top_p | 0.95 | 0.8 |
| 适用场景 | 复杂推理、数学题、逻辑问题 | 简单问答、闲聊 |
| 响应速度 | 较慢（需要生成思考过程） | 较快 |

### 什么时候用思考模式？
- 需要复杂推理的问题（如数学证明）
- 需要多步骤解决的问题
- 想了解模型的"思路"时

### 什么时候关闭思考模式？
- 简单的问答对话
- 需要快速响应时
- 不需要看到推理过程时

In [12]:
# ============================================================
# 第六步：关闭思考模式测试
# ============================================================
#
# 这个 Cell 演示如何关闭思考模式
# 代码结构与前面相同，只是参数不同
# ============================================================

# 【定义新的问题】
prompt2 = "你是谁？"
# 使用不同的变量名（prompt2）避免覆盖之前的变量

# 【构建消息列表】
messages2 = [{"role": "user", "content": prompt2}]

# ============================================================
# 【格式化输入 - 关闭思考模式】
# ============================================================

text2 = tokenizer.apply_chat_template(
    messages2,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False  # ⭐ 关键区别：设为 False 关闭思考模式
)
#
# enable_thinking=False 的效果：
# - 模型不会生成 <think>...</think> 部分
# - 直接输出最终答案
# - 响应速度更快
# - 输出更简洁

# ============================================================
# 【编码并移动到 GPU】
# ============================================================

model_inputs2 = tokenizer([text2], return_tensors="pt").to(model.device)
# 与之前相同的处理流程

# ============================================================
# 【生成 - 使用非思考模式推荐参数】
# ============================================================

generated_ids2 = model.generate(
    **model_inputs2,
    max_new_tokens=512,     # 非思考模式不需要太长，因为没有推理过程
    temperature=0.7,        # ⭐ 非思考模式推荐：0.7（比思考模式略高）
    top_p=0.8,              # ⭐ 非思考模式推荐：0.8（比思考模式略低）
    top_k=20,               # 保持不变
    do_sample=True
)
#
# 【参数调整说明】
# 非思考模式的参数与思考模式略有不同：
#
# | 参数 | 思考模式 | 非思考模式 | 原因 |
# |------|---------|-----------|------|
# | temperature | 0.6 | 0.7 | 非思考模式可以稍微更随机一些 |
# | top_p | 0.95 | 0.8 | 非思考模式采样范围可以更小 |
# | max_new_tokens | 2048 | 512 | 不需要生成思考过程，长度更短 |
#
# 这些是官方推荐的参数，可以根据实际效果微调

# ============================================================
# 【提取生成结果并解码】
# ============================================================

output_ids2 = generated_ids2[0][len(model_inputs2.input_ids[0]):].tolist()
# 同样的切片操作：去掉输入部分，只保留新生成的

response2 = tokenizer.decode(output_ids2, skip_special_tokens=True)
# 解码为文本

# ============================================================
# 【输出结果】
# ============================================================

print("=" * 50)
print("模型响应（无思考过程）：")
print("=" * 50)
print(response2)

# ============================================================
# 【对比思考模式的输出】
#
# 思考模式输出（前面的例子）：
# <think>
# 好的，用户让我提供一个关于大模型的简短介绍...
# （很长的思考过程）
# </think>
# 大模型（Large Model）是指参数量巨大...
#
# 非思考模式输出（这个例子）：
# 我是通义千问，由通义实验室研发的超大规模语言模型...
# （直接是答案，没有思考过程）
#
# 可以看到非思考模式的输出更加简洁直接
# ============================================================

模型响应（无思考过程）：
我是通义千问，是由通义实验室开发的超大规模语言模型。我能够进行多轮对话，回答各种问题，创作文字，编写程序，甚至进行推理和决策。我的目标是成为人类最聪明的助手，帮助人类更好地生活和工作。


---

## 学习总结

### 本教程学到的内容

1. **模型下载**
   - 使用 ModelScope 从国内服务器下载模型
   - `snapshot_download()` 函数的使用

2. **模型加载**
   - 使用 Transformers 库加载模型
   - `AutoTokenizer` 和 `AutoModelForCausalLM` 的使用
   - `torch_dtype=torch.float16` 节省显存
   - `device_map="auto"` 自动分配设备

3. **文本生成**
   - `apply_chat_template()` 格式化对话
   - `model.generate()` 生成文本
   - 理解各种采样参数（temperature、top_p、top_k）

4. **Qwen3 特性**
   - 思考模式（enable_thinking）的使用
   - 思考模式与非思考模式的参数差异

### 核心代码模板

```python
# 1. 加载
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.float16, device_map="auto")

# 2. 格式化
messages = [{"role": "user", "content": "你的问题"}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# 3. 生成
inputs = tokenizer([text], return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.7, do_sample=True)

# 4. 解码
response = tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True)
```

### 下一步学习建议

1. 尝试修改采样参数，观察输出变化
2. 尝试多轮对话（在 messages 中添加多条消息）
3. 学习 vLLM 部署（使用其他平台如 Kaggle/AutoDL）
4. 学习模型微调（Fine-tuning）

---

## 附录：Python 语法速查

### 本教程用到的 Python 语法

| 语法 | 示例 | 说明 |
|------|------|------|
| **变量赋值** | `x = 10` | 不需要声明类型 |
| **字符串** | `"hello"` 或 `'hello'` | 单引号双引号都可以 |
| **列表** | `[1, 2, 3]` | 有序集合，用方括号 |
| **字典** | `{"key": "value"}` | 键值对，用花括号 |
| **导入** | `import torch` | 导入整个模块 |
| **部分导入** | `from x import y` | 只导入特定内容 |
| **函数调用** | `func(arg1, arg2)` | 位置参数 |
| **关键字参数** | `func(name="value")` | 指定参数名 |
| **字典解包** | `func(**dict)` | 将字典展开为参数 |
| **切片** | `list[1:3]` | 取索引 1 到 2 的元素 |
| **方法链** | `obj.method1().method2()` | 连续调用方法 |
| **f-string** | `f"值是{x}"` | 格式化字符串 |
| **字符串重复** | `"=" * 50` | 重复字符串 |

### 常见缩写含义

| 缩写 | 全称 | 含义 |
|------|------|------|
| LLM | Large Language Model | 大语言模型 |
| NLP | Natural Language Processing | 自然语言处理 |
| GPU | Graphics Processing Unit | 图形处理单元（用于加速计算） |
| CUDA | Compute Unified Device Architecture | NVIDIA 的并行计算平台 |
| API | Application Programming Interface | 应用程序接口 |
| token | - | 文本的最小单位（词或子词） |
| tensor | - | 张量，多维数组 |